# 🚢 Titanic Survival Prediction - Complete Analysis
## Comprehensive Machine Learning Project

This notebook provides a complete analysis of the Titanic dataset, from exploratory data analysis to model building and evaluation.

## 1. Setup and Data Loading

In [ ]:
import sys
import os
sys.path.append('src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Load data
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print(f"Training data shape: {train.shape}")
print(f"Test data shape: {test.shape}")
print(f"\nTraining columns:\n{train.columns.tolist()}")

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Basic information
print("=== TRAINING DATA INFO ===")
print(train.info())
print("\n=== MISSING VALUES ===")
print(train.isnull().sum())

# Display first few rows
print("\n=== SAMPLE DATA ===")
display(train.head())

In [ ]:
# Statistical summary
print("=== STATISTICAL SUMMARY ===")
display(train.describe())

# Survival distribution
print(f"\nSurvival Rate: {train['Survived'].mean()*100:.2f}%")
print(f"Male Survival Rate: {train[train['Sex']=='male']['Survived'].mean()*100:.2f}%")
print(f"Female Survival Rate: {train[train['Sex']=='female']['Survived'].mean()*100:.2f}%")


## 3. Data Visualization

In [ ]:
from visualization import TitanicVisualizer

# Create visualizer instance
viz = TitanicVisualizer()

# Create EDA dashboard
eda_fig = viz.create_eda_dashboard(train, target='Survived')
plt.show()

## 4. Data Preprocessing and Feature Engineering

In [ ]:
from preprocessing import TitanicPreprocessor

# Create preprocessor
preprocessor = TitanicPreprocessor()

# Preprocess data
train_processed = preprocessor.preprocess_train(train)
test_processed = preprocessor.preprocess_train(test)

print(f"Original training shape: {train.shape}")
print(f"Processed training shape: {train_processed.shape}")
print(f"\nNew features created:")
new_features = [col for col in train_processed.columns if col not in train.columns]
print(new_features)

## 5. Model Training and Evaluation

In [ ]:
from model_training import TitanicModelTrainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Encode and scale features
train_encoded = preprocessor.encode_features(train_processed, is_train=True)
test_encoded = preprocessor.encode_features(test_processed, is_train=False)

train_scaled = preprocessor.scale_features(train_encoded, is_train=True)
test_scaled = preprocessor.scale_features(test_encoded, is_train=False)

# Prepare data for modeling
X = train_scaled.drop('Survived', axis=1)
y = train_scaled['Survived']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")

In [ ]:
# Initialize and train models
trainer = TitanicModelTrainer(random_state=42)
trainer.initialize_models()

# Train all models
results_df = trainer.train_models(X_train, y_train, X_val, y_val)

# Display results
print("\n=== MODEL PERFORMANCE SUMMARY ===")
display(results_df.sort_values('accuracy', ascending=False))

## 6. Model Visualization

In [ ]:
# Create model performance visualization
viz.create_model_performance_chart(results_df)
plt.show()

# Create ROC curves
viz.create_roc_curves(trainer.models, X_val, y_val)
plt.show()

# Create confusion matrix for best model
y_pred_best = trainer.best_model.predict(X_val)
viz.create_confusion_matrix_heatmap(y_val, y_pred_best, trainer.best_model_name)
plt.show()

# Feature importance
viz.create_feature_importance_chart(trainer.best_model, X.columns.tolist())
plt.show()

## 7. Hyperparameter Tuning

In [ ]:
# Perform hyperparameter tuning
best_model = trainer.hyperparameter_tuning(X_train, y_train)

# Evaluate tuned model
y_pred_tuned = best_model.predict(X_val)
tuned_accuracy = accuracy_score(y_val, y_pred_tuned)
print(f"Tuned model accuracy: {tuned_accuracy:.4f}")

## 8. Final Model and Predictions

In [ ]:
# Train final model on all data
final_model = trainer.best_model
final_model.fit(X, y)

# Make predictions
test_predictions = final_model.predict(test_scaled)

# Create submission
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': test_predictions
})

# Save submission
submission.to_csv('titanic_predictions.csv', index=False)
print(f"Submission saved. Predicted survival rate: {test_predictions.mean()*100:.2f}%")

## 9. Export Results to Excel

In [ ]:
# Create Excel report
from openpyxl import Workbook

# Create workbook
wb = Workbook()
ws = wb.active
ws.title = "Model Results"

# Write headers
ws.append(["Model", "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"])

# Write results
for _, row in results_df.iterrows():
    ws.append([
        row['model'],
        row['accuracy'],
        row.get('precision', 'N/A'),
        row.get('recall', 'N/A'),
        row.get('f1_score', 'N/A'),
        row.get('roc_auc', 'N/A')
    ])

# Save workbook
excel_path = "titanic_model_results.xlsx"
wb.save(excel_path)
print(f"Excel report saved to {excel_path}")

## 10. Summary and Key Insights

In [ ]:
print("=" * 60)
print("KEY INSIGHTS FROM TITANIC ANALYSIS")
print("=" * 60)

print("\n1. DEMOGRAPHIC INSIGHTS:")
print(f"   - Overall survival rate: {train['Survived'].mean()*100:.1f}%")
print(f"   - Female survival rate: {train[train['Sex']=='female']['Survived'].mean()*100:.1f}%")
print(f"   - Male survival rate: {train[train['Sex']=='male']['Survived'].mean()*100:.1f}%")

print("\n2. SOCIO-ECONOMIC INSIGHTS:")
for pclass in sorted(train['Pclass'].unique()):
    survival_rate = train[train['Pclass'] == pclass]['Survived'].mean() * 100
    print(f"   - Class {pclass} survival rate: {survival_rate:.1f}%")

print("\n3. MODELING INSIGHTS:")
print(f"   - Best model: {trainer.best_model_name}")
print(f"   - Best accuracy: {results_df['accuracy'].max():.4f}")
print(f"   - Number of features used: {X.shape[1]}")

print("\n4. RECOMMENDATIONS:")
print("   - Focus on gender and class as primary predictors")
print("   - Consider family size in evacuation planning")
print("   - Age should be considered, especially for children")
print("   - Fare price correlates with survival probability")